# UNO Game AI - Assignment 2
### AI2002 Artificial Intelligence | Ms. Umarah Qaseem

| Player | Algorithm | Strategy |
|--------|-----------|----------|
| P1 | Minimax | Defensive |
| P2 | Expectimax | Offensive |
| P3 | Minimax / Manual | Simulation or User |

**GitHub Repository:** https://github.com/hasnain-afkar/UNO-GameAI  

In [2]:
import random
import copy


## 1. Card Class

In [3]:
# Represents one UNO card
# Color: Red, Blue, Green, Yellow
# Value: 0-9 or Skip
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

    # Print card nicely e.g. Red 5
    def __repr__(self):
        return str(self.color) + ' ' + str(self.value)

    # Compare two cards for equality
    def __eq__(self, other):
        return self.color == other.color and self.value == other.value

    # Allow cards to be used in sets and dicts
    def __hash__(self):
        return hash((self.color, self.value))


## 2. Deck Generator

In [4]:
# Creates a full simplified UNO deck and shuffles it
# Each color: numbers 0-9 (two copies) + 2 Skip cards
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []
    # Two copies of each number card per color
    for color in colors:
        for num in range(10):
            deck.append(Card(color, num))
            deck.append(Card(color, num))
    # Two Skip cards per color
    for color in colors:
        deck.append(Card(color, 'Skip'))
        deck.append(Card(color, 'Skip'))
    random.shuffle(deck)
    return deck

deck = generate_deck()
print('Deck size:', len(deck))
print('Sample cards:', deck[:5])


Deck size: 88
Sample cards: [Red 3, Yellow 0, Green 8, Green 6, Green 5]


## 3. Legal Move Generator

In [5]:
# Returns cards from hand that can legally be played on top_card
# Rule: card must match top card color OR value
def get_valid_moves(hand, top_card):
    valid = []
    for card in hand:
        if card.color == top_card.color or card.value == top_card.value:
            valid.append(card)
    return valid


## 4. State Transition and Game Setup

### State Representation
The game state dictionary as required by the assignment:

```python
state = {
    'p1'        : ai_cards,          # Player 1 hand (Minimax)
    'p2'        : opponent1_cards,   # Player 2 hand (Expectimax)
    'p3'        : opponent2_cards,   # Player 3 hand (Manual/Sim)
    'top_card'  : top_card,
    'deck'      : deck,
    'skip_next' : False
}
```

In [6]:
# Applies a move and returns the updated state (deep copy)
# card=None means the player draws one card from the deck
def apply_move(state, player_key, card):
    new_state = copy.deepcopy(state)
    if card is None:
        # Draw rule: take one card from top of deck
        if new_state['deck']:
            drawn = new_state['deck'].pop(0)
            new_state[player_key].append(drawn)
        new_state['skip_next'] = False
    else:
        # Play rule: remove from hand and place on pile
        new_state[player_key].remove(card)
        new_state['top_card'] = card
        # Skip rule: set flag if Skip card was played
        new_state['skip_next'] = (card.value == 'Skip')
    return new_state


# Sets up a brand new game state
def setup_game():
    deck = generate_deck()
    # Deal 5 cards each as required by assignment
    p1 = [deck.pop() for _ in range(5)]
    p2 = [deck.pop() for _ in range(5)]
    p3 = [deck.pop() for _ in range(5)]
    # Starting top card must not be a Skip
    top = deck.pop()
    while top.value == 'Skip':
        deck.insert(0, top)
        top = deck.pop()
    return {
        'p1': p1,         # ai_cards
        'p2': p2,         # opponent1_cards
        'p3': p3,         # opponent2_cards
        'top_card': top,
        'deck': deck,
        'skip_next': False
    }

state = setup_game()
print('Top card:', state['top_card'])
print('P1 hand:', state['p1'])
print('P2 hand:', state['p2'])
print('P3 hand:', state['p3'])
print('Valid moves for P1:', get_valid_moves(state['p1'], state['top_card']))


Top card: Red 2
P1 hand: [Green 7, Blue 8, Red 5, Yellow 1, Yellow 8]
P2 hand: [Yellow 4, Green 8, Green 6, Yellow 8, Yellow 5]
P3 hand: [Blue 9, Blue 5, Blue 4, Yellow 4, Green 8]
Valid moves for P1: [Red 5]


## 5. Evaluation Function

**Assignment Formula:** `Score = 50 - 5(C_AI) + 2(C_opp) + 3(S)`

Weights tuned differently per strategy as required:

| Weight | Defensive P1/P3 | Offensive P2 | Reason |
|--------|-----------------|--------------|--------|
| w_ai   | 5               | 6            | Offensive penalises own cards harder |
| w_opp  | 2               | 2            | Same for both |
| w_skip | 4               | 2            | Defensive holds Skip cards more |

In [7]:
# Scores the state from ai_key's perspective
# strategy='defensive' for P1/P3, 'offensive' for P2
def evaluate(state, ai_key, opp_keys, strategy='defensive'):
    ai_hand = state[ai_key]
    opp_hands = [state[k] for k in opp_keys]
    # C_AI: cards in hand
    c_ai = len(ai_hand)
    # C_opp: average opponent cards
    c_opp = sum(len(h) for h in opp_hands) / max(len(opp_hands), 1)
    # S: Skip cards in hand
    s = sum(1 for card in ai_hand if card.value == 'Skip')
    if strategy == 'defensive':
        w_ai, w_opp, w_skip = 5, 2, 4
    else:
        w_ai, w_opp, w_skip = 6, 2, 2
    # Apply assignment formula with tuned weights
    return 50 - w_ai * c_ai + w_opp * c_opp + w_skip * s

print('P1 defensive score:', evaluate(state, 'p1', ['p2', 'p3'], 'defensive'))
print('P2 offensive score:', evaluate(state, 'p2', ['p1', 'p3'], 'offensive'))


P1 defensive score: 35.0
P2 offensive score: 30.0


## 6. Minimax - Player 1 (Defensive)
- MAX node: AI maximises its score
- MIN node: Opponent minimises AI score
- Depth: 3

In [8]:
# Minimax search for Player 1 - defensive strategy
# Returns (best_score, best_card) - card=None means draw
def minimax(state, depth, is_maximising, ai_key, opp_keys):
    # Win condition: AI emptied hand
    if len(state[ai_key]) == 0:
        return (1000, None)
    # Loss condition: opponent emptied hand
    for opp in opp_keys:
        if len(state[opp]) == 0:
            return (-1000, None)
    # Depth limit: evaluate current state
    if depth == 0:
        return (evaluate(state, ai_key, opp_keys, 'defensive'), None)

    # MAX node: AI picks best move
    if is_maximising:
        best_score = float('-inf')
        best_card = None
        valid = get_valid_moves(state[ai_key], state['top_card'])
        moves = valid if valid else [None]  # must draw if no valid moves
        for card in moves:
            ns = apply_move(state, ai_key, card)
            sc, _ = minimax(ns, depth - 1, False, ai_key, opp_keys)
            if sc > best_score:
                best_score = sc
                best_card = card
        return (best_score, best_card)

    # MIN node: opponent picks worst move for AI
    else:
        best_score = float('inf')
        opp = opp_keys[0]
        valid = get_valid_moves(state[opp], state['top_card'])
        moves = valid if valid else [None]
        for card in moves:
            ns = apply_move(state, opp, card)
            sc, _ = minimax(ns, depth - 1, True, ai_key, opp_keys)
            if sc < best_score:
                best_score = sc
        return (best_score, None)

sc, mv = minimax(state, 3, True, 'p1', ['p2', 'p3'])
print('Minimax -> Play:', mv if mv else 'DRAW', '| Score:', round(sc, 1))


Minimax -> Play: Red 5 | Score: 44.0
